In [0]:
%run
"/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

In [0]:
from pyspark.sql.functions import col

path = f'/Volumes/databox/juridico_comum/arquivos/financeiro/bases_escritorios/'

nome_arquivo = 'Painel Despesas Jurídicas.xlsx' 

caminho_arquivo = path + nome_arquivo

print(caminho_arquivo)

In [0]:
/Volumes/databox/juridico_comum/arquivos/financeiro/RNO - Origem/Composicao Mes/RNO_202505.xlsxspark.conf.set("spark.sql.caseSensitive", "true")
abas_usadas = ['Base Real x Orçado','Base Razão','Validação','Notas Provisionadas']

colunas_tipos = {
    "Nota Fiscal": "string",
}

for aba in abas_usadas:
    if aba == 'Base Real x Orçado':
        df_real_vs_orca = read_excel(caminho_arquivo,f"'{aba}'!A1")

        df_real_vs_orca = adjust_column_names(df_real_vs_orca)
    if aba == 'Base Razão':
        df_razao = read_excel(caminho_arquivo,f"'{aba}'!A2")
        df_razao = df_razao.drop("Ano","Fornecedor")

        for c, t in colunas_tipos.items():
            df_razao = df_razao.withColumn(c, col(c).cast(t))

        df_razao = adjust_column_names(df_razao)
    if aba == 'Validação':
        df_validacao = read_excel(caminho_arquivo,f"'{aba}'!A2")
        df_validacao = adjust_column_names(df_validacao)
    if aba == 'Notas Provisionadas':
        df_notas_provisorias = read_excel(caminho_arquivo,f"'{aba}'!S2")
        df_notas_provisorias = adjust_column_names(df_notas_provisorias)

# Converte para data somando a data base do Excel
df_real_vs_orca = df_real_vs_orca.withColumn("MÊS_COMPETÊNCIA", col("MÊS_COMPETÊNCIA").cast("int"))
df_real_vs_orca = df_real_vs_orca.withColumn("MÊS_DA_MIGO", col("MÊS_DA_MIGO").cast("int"))
df_real_vs_orca = df_real_vs_orca.withColumn("DATA_PAGAMENTO", col("DATA_PAGAMENTO").cast("int"))

df_real_vs_orca = df_real_vs_orca.withColumn("MÊS_COMPETÊNCIA", expr("date_add(to_date('1899-12-30'), `MÊS_COMPETÊNCIA`)"))
df_real_vs_orca = df_real_vs_orca.withColumn("MÊS_DA_MIGO", expr("date_add(to_date('1899-12-30'), `MÊS_DA_MIGO`)"))
df_real_vs_orca = df_real_vs_orca.withColumn("DATA_PAGAMENTO", expr("date_add(to_date('1899-12-30'), DATA_PAGAMENTO)"))


# Escreve tabelas deltas para consumo no power bi
df_real_vs_orca.write.format("delta") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"databox.juridico_comum.df_escritorios_real_vs_orca_teste")

df_razao.write.format("delta") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"databox.juridico_comum.df_escritorios_razao_teste")

df_validacao.write.format("delta") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"databox.juridico_comum.df_escritorios_validacao_teste")

df_notas_provisorias.write.format("delta") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"databox.juridico_comum.df_escritorios_notas_provisorias_teste")

In [0]:
print(f'Pode usar')